**Configuration note:** this notebook verifies that contaminant contigs flagged by NCBI FCS-GX / FCS-adapter (run on the NCBI Galaxy portal) did not land in the final chromosome scaffolds. The input paths below mirror `config.sh` at the repo root — edit them for your own data. The FCS reports (`contamination_action.txt`, `adaptor_report.txt`) are read from this working directory.


In [ ]:
# Input paths — edit for your environment (mirror config.sh at the repo root).
HAPHIC_BUILD_DIR = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/output/outputs-from-haphic-alignment/references_hifiasm_male403_hifiHiCMode_041425_trail1_allChrom/04.build"
fasta_file = f"{HAPHIC_BUILD_DIR}/assembly_final.fasta"
agp_file    = f"{HAPHIC_BUILD_DIR}/out_JBAT_review_with_orig_name.agp"


## Goal of this notebook
Check if any of the contaminating contigs are found made it into the scaffold

In [1]:
import pandas as pd

In [22]:
fasta_file = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/output/outputs-from-haphic-alignment/references_hifiasm_male403_hifiHiCMode_041425_trail1_allChrom/04.build/assembly_final.fasta"

# Read all lines, keep only those starting with '>'
with open(fasta_file, 'r') as f:
    # headers = [line.strip() for line in f if line.startswith('>')]
    headers = [line[1:].strip() for line in f if line.startswith('>')]

# Convert to DataFrame (optional, if you need tabular structure)
df_headers = pd.DataFrame(headers, columns=['header'])
print(df_headers)

                                           header
0    PGA_scaffold_1__19_contigs__length_197914338
1    PGA_scaffold_2__14_contigs__length_185300982
2     PGA_scaffold_3__8_contigs__length_152493930
3     PGA_scaffold_4__9_contigs__length_151562733
4    PGA_scaffold_5__10_contigs__length_142911117
..                                            ...
283      PGA_scaffold_284__1_contigs__length_5739
284      PGA_scaffold_285__1_contigs__length_5604
285      PGA_scaffold_286__1_contigs__length_5454
286      PGA_scaffold_287__1_contigs__length_4544
287      PGA_scaffold_288__1_contigs__length_3924

[288 rows x 1 columns]


In [23]:
df_contam = pd.read_csv("contamination_action.txt", sep="\t", skiprows=1, skipfooter=1, engine='python')
contam = df_contam["#seq_id"].str.replace("lcl|", "", regex=False)
# contam =["ptg000006l"]

contam

0     ptg000355l
1     ptg000374l
2     ptg000375l
3     ptg000377l
4     ptg000378l
         ...    
84    ptg000488l
85    ptg000489l
86    ptg000490l
87    ptg000491l
88    ptg000492l
Name: #seq_id, Length: 89, dtype: object

In [24]:
df_scaff = pd.read_csv("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/output/outputs-from-haphic-alignment/references_hifiasm_male403_hifiHiCMode_041425_trail1_allChrom/04.build/out_JBAT_review_with_orig_name.agp",sep="\t",skiprows=2, names=["scaffold_name","start","end","contig_num","type","contig_name","fillter","length","type2"])
df_scaff = df_scaff[["scaffold_name","contig_name"]]
df_scaff = df_scaff[df_scaff["contig_name"]!="100"]
df_scaff

,scaffold_name,contig_name
0,PGA_scaffold_1__19_contigs__length_197914338,ptg000007l
2,PGA_scaffold_1__19_contigs__length_197914338,ptg000175l
4,PGA_scaffold_1__19_contigs__length_197914338,ptg000073l
6,PGA_scaffold_1__19_contigs__length_197914338,ptg000131l
8,PGA_scaffold_1__19_contigs__length_197914338,ptg000040l
...,...,...
689,PGA_scaffold_284__1_contigs__length_5739,ptg000524l
690,PGA_scaffold_285__1_contigs__length_5604,ptg000481l
691,PGA_scaffold_286__1_contigs__length_5454,ptg000445l
692,PGA_scaffold_287__1_contigs__length_4544,ptg000484l


Get the id for the first 30 chromosome

In [25]:
# df_headers = df_headers[0:30]
# df_headers = df_headers[0:288]
df_headers

,header
0,PGA_scaffold_1__19_contigs__length_197914338
1,PGA_scaffold_2__14_contigs__length_185300982
2,PGA_scaffold_3__8_contigs__length_152493930
3,PGA_scaffold_4__9_contigs__length_151562733
4,PGA_scaffold_5__10_contigs__length_142911117
...,...
283,PGA_scaffold_284__1_contigs__length_5739
284,PGA_scaffold_285__1_contigs__length_5604
285,PGA_scaffold_286__1_contigs__length_5454
286,PGA_scaffold_287__1_contigs__length_4544


Select for the contig_name within the chromosome

In [26]:
df_scaff = df_scaff[df_scaff["scaffold_name"].isin(df_headers["header"])]
df_scaff

,scaffold_name,contig_name
0,PGA_scaffold_1__19_contigs__length_197914338,ptg000007l
2,PGA_scaffold_1__19_contigs__length_197914338,ptg000175l
4,PGA_scaffold_1__19_contigs__length_197914338,ptg000073l
6,PGA_scaffold_1__19_contigs__length_197914338,ptg000131l
8,PGA_scaffold_1__19_contigs__length_197914338,ptg000040l
...,...,...
689,PGA_scaffold_284__1_contigs__length_5739,ptg000524l
690,PGA_scaffold_285__1_contigs__length_5604,ptg000481l
691,PGA_scaffold_286__1_contigs__length_5454,ptg000445l
692,PGA_scaffold_287__1_contigs__length_4544,ptg000484l


 See if any of the contam is within the contig_name 

In [27]:
df_scaff["contam"]=df_scaff["contig_name"].isin(contam)

In [28]:
df_scaff["contam"].value_counts()

contam
False    402
True      89
Name: count, dtype: int64

In [17]:
df_scaff[df_scaff["contig_name"].isin(contam)]

,scaffold_name,contig_name,contam
242,PGA_scaffold_15__3_contigs__length_121653570,ptg000006l,True
